In [ ]:
import boto3

# Assume into OrganizationAccountAccessRole in a target account
# By default this uses your *current* account ID; change TARGET_ACCOUNT_ID
# if you want to assume into a different AWS account.
base_session = boto3.session.Session()
sts = base_session.client("sts")
current_identity = sts.get_caller_identity()

TARGET_ACCOUNT_ID = 396511695522  # TODO: set to another account ID if needed
ROLE_NAME = "OrganizationAccountAccessRole"
ROLE_ARN = f"arn:aws:iam::{TARGET_ACCOUNT_ID}:role/{ROLE_NAME}"

print("Assuming role:", ROLE_ARN)
assumed = sts.assume_role(
    RoleArn=ROLE_ARN,
    RoleSessionName="rgasa-purge-session",
)
creds = assumed["Credentials"]

# Create a session with the assumed-role credentials
assumed_session = boto3.session.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Optionally make the assumed role the default for future boto3.client/resource calls
boto3.setup_default_session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Confirm who we are now
identity = assumed_session.client("sts").get_caller_identity()

print("AWS STS get_caller_identity() after assume-role:")
print(f"  Account:   {identity['Account']}")
print(f"  UserId:    {identity['UserId']}")
print(f"  ARN:       {identity['Arn']}")
print(f"  Region:    {assumed_session.region_name}")

print("You can now use boto3.client(...) and it will use the assumed role.")

Assuming role: arn:aws:iam::396511695522:role/OrganizationAccountAccessRole
AWS STS get_caller_identity() after assume-role:
  Account:   396511695522
  UserId:    AROAJ5YTGMWHKRNBYDR4Y:rgasa-purge-session
  ARN:       arn:aws:sts::396511695522:assumed-role/OrganizationAccountAccessRole/rgasa-purge-session
  Region:    eu-west-1
You can now use boto3.client(...) and it will use the assumed role.


In [33]:
import boto3
from datetime import datetime, timezone

BUCKET_NAME = "comotion-comodash-rgasa-datainput"
START = datetime(2025, 12, 8, 7, 50, 0, tzinfo=timezone.utc)
END = datetime(2025, 12, 8, 7, 55, 0, tzinfo=timezone.utc)

PREFIXES = ["inforce/", "terminations/", "treaty_info/"]

s3 = boto3.client("s3")

matched_objects = []

for prefix in PREFIXES:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=prefix):
        for obj in page.get("Contents", []):
            last_modified = obj["LastModified"]
            if START <= last_modified <= END:
                matched_objects.append({
                    "Key": obj["Key"],
                    "LastModified": last_modified,
                    "Size": obj["Size"],
                })

print(f"Found {len(matched_objects)} objects in {BUCKET_NAME} between {START} and {END} (UTC)")
for o in matched_objects:
    print(f"{o['LastModified'].isoformat()}\t{o['Size']}\t{o['Key']}")

Found 37 objects in comotion-comodash-rgasa-datainput between 2025-12-08 07:50:00+00:00 and 2025-12-08 07:55:00+00:00 (UTC)
2025-12-08T07:53:40+00:00	1537990	inforce/data_import_batch=2025-12-08/service_client_id=0/02485930-d40b-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:53:45+00:00	1599478	inforce/data_import_batch=2025-12-08/service_client_id=0/04fd4000-d40b-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:53:50+00:00	1616954	inforce/data_import_batch=2025-12-08/service_client_id=0/07cd7700-d40b-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:53:55+00:00	1540978	inforce/data_import_batch=2025-12-08/service_client_id=0/0ac53330-d40b-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:53:59+00:00	1559694	inforce/data_import_batch=2025-12-08/service_client_id=0/0d8c4270-d40b-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:54:04+00:00	1605429	inforce/data_import_batch=2025-12-08/service_client_id=0/1062bb00-d40b-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:54:09+00:00	1643299	inforce/data_import_batch=2025-1